## Multiple Linear Regression

It is a method for modelling relationships b/w variables. We can infer a variable based on other variables. You take more than one _x_ (independent variables) and predict a single _y_ (dependent variable). Here, the dependent variable is a metric. 

However, there are a lot of different factors that need to be taken into consideration:
* The weight terms or $X$ is no longer just a single value, but a vector of features
* $y$ remains the same as it is still predicting a single dependent variable
* When computing the formula: $\hat{y} = m_0 + m_1 \cdot x_1 + ... + m_n \cdot x_n + b$, then we need to multiply the matrix of features, i.e., $X$ with the vector of weights $W$, and add $b$ (the bias term)
* There is also a different method for calculating the weights


In [47]:
import sklearn as sk
import numpy as np
import torch
import statsmodels.api as sm
import jax
import jaxlib
import plotly.express as px

import pandas as pd
import polars as pl

In [2]:
def check_homoscedasticity(model_preds: np.ndarray,
                           y_test: np.ndarray):
        residuals = y_test - model_preds

        figure = px.scatter(x=model_preds, y=residuals,
                            labels={"x": "Model predictions", "y": "Residuals/Errors"},
                            title="Residual plot")
        figure.add_hline(y=0, line_dash="dash", line_color="red")
        figure.show()

In [3]:
from ucimlrepo import fetch_ucirepo 

df_uci = fetch_ucirepo(id=477) 
df_pd = df_uci.data.original
df = pl.from_pandas(df_pd)

In [4]:
df

No,X1 transaction date,X2 house age,X3 distance to the nearest MRT station,X4 number of convenience stores,X5 latitude,X6 longitude,Y house price of unit area
i64,f64,f64,f64,i64,f64,f64,f64
1,2012.917,32.0,84.87882,10,24.98298,121.54024,37.9
2,2012.917,19.5,306.5947,9,24.98034,121.53951,42.2
3,2013.583,13.3,561.9845,5,24.98746,121.54391,47.3
4,2013.5,13.3,561.9845,5,24.98746,121.54391,54.8
5,2012.833,5.0,390.5684,5,24.97937,121.54245,43.1
…,…,…,…,…,…,…,…
410,2013.0,13.7,4082.015,0,24.94155,121.50381,15.4
411,2012.667,5.6,90.45606,9,24.97433,121.5431,50.0
412,2013.25,18.8,390.9696,7,24.97923,121.53986,40.6


In [5]:
df = df.drop('No')

In [6]:
df

X1 transaction date,X2 house age,X3 distance to the nearest MRT station,X4 number of convenience stores,X5 latitude,X6 longitude,Y house price of unit area
f64,f64,f64,i64,f64,f64,f64
2012.917,32.0,84.87882,10,24.98298,121.54024,37.9
2012.917,19.5,306.5947,9,24.98034,121.53951,42.2
2013.583,13.3,561.9845,5,24.98746,121.54391,47.3
2013.5,13.3,561.9845,5,24.98746,121.54391,54.8
2012.833,5.0,390.5684,5,24.97937,121.54245,43.1
…,…,…,…,…,…,…
2013.0,13.7,4082.015,0,24.94155,121.50381,15.4
2012.667,5.6,90.45606,9,24.97433,121.5431,50.0
2013.25,18.8,390.9696,7,24.97923,121.53986,40.6


In [7]:
df.columns

['X1 transaction date',
 'X2 house age',
 'X3 distance to the nearest MRT station',
 'X4 number of convenience stores',
 'X5 latitude',
 'X6 longitude',
 'Y house price of unit area']

In [8]:
df_X = df.select(['X1 transaction date', 'X2 house age', 'X3 distance to the nearest MRT station', 'X4 number of convenience stores', 'X5 latitude', 'X6 longitude'])
df_y = df.select('Y house price of unit area')

In [9]:
df_X.shape, df_y.shape

((414, 6), (414, 1))

In [10]:
df_y

Y house price of unit area
f64
37.9
42.2
47.3
54.8
43.1
…
15.4
50.0
40.6


In [11]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from plotly_resampler import FigureResampler

In [12]:
fig = make_subplots(rows=2, cols=3, 
                    subplot_titles=df_X.columns,
                    horizontal_spacing=0.1, vertical_spacing=0.1)

for i, col_name in enumerate(df_X.columns):
    row = (i // 3) + 1
    col = (i % 3) + 1

    fig.add_trace(
        go.Scattergl(
            x=df_X[col_name],
            y =df_y['Y house price of unit area'],
            mode="markers",
            marker=dict(size=5, color='blue', opacity=0.5),
            name=col_name
        ),
        row=row, col=col
    )

fig.update_layout(
    title_text="Features vs House Price of all the features in the dataset",
    height=800, width=1000,
    showlegend=False)
fig.show()

Formula to compute gradients in multiple linear regression:

$dW = \frac{1}{N} X^T (\hat{Y} - Y)$

$dB = \frac{1}{N} \sum \epsilon$

In [ ]:
class MLR1:
    def __init__(self, learning_rate: int = 0.01, n_iters: int = 100):
        self.learning_rate = learning_rate
        self.n_iters = n_iters
        self.weights = None
        self.bias = None
        self.losses = []

    def mse(self, y_true: np.ndarray, y_pred: np.ndarray) -> float:
        return np.mean((y_pred - y_true) ** 2)

    def fit(self, X: np.ndarray, y: np.ndarray):
        """This method trains the model using gradient descent
        to minimize the mean square error loss function.

        Args:
            X (np.ndarray): This is the vector of training features
            y (np.ndarray): This is the target variable
        """

        # here we get the number of samples and numeber of features in the training data
        n_samples, n_features = X.shape
        #initialize the weights vector to random values of size n_features and the bias to zero
        self.weights = np.zeros(n_features)
        self.bias = 0

        for epoch in range(self.n_iters):
            # perform matrix multiplication of the input features X with the weights and add the bias to get the predicted values
            y_pred = np.dot(X, self.weights) + self.bias

            # compute the loss and append it to the list of losses for each epocj
            loss = self.mse(y, y_pred)
            self.losses.append(loss)

            # compute the residual/errors 
            errors = y_pred - y
            # compute the gradients of the loss with respect of the weights and bias
            dw = (1 / n_samples) * np.dot(X.T, (y_pred - y))
            db = (1 / n_samples) * np.sum(errors)

            # update the weights and bias using the computed gradients and the learning rate
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db

            if epoch % 10 == 0:
                print(f"Epoch: {epoch}, loss: {loss}")

    def predict(self, X: np.ndarray):
        return np.dot(X, self.weights) + self.bias

In [35]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df_X.to_numpy(),
                                                    df_y.to_numpy(),
                                                    test_size=0.2,
                                                    random_state=42)

X_train.shape, y_train.shape, X_test.shape, y_test.shape

((331, 6), (331, 1), (83, 6), (83, 1))

In [ ]:
mlr1 = MLR1(0.1, 100)

"""Here, it is the case that when you pass y directly, 
it is of shape (331, 1), making it a 2D array. And since you are targeting 
a 1D array for the target variable, you need to flatten the array first
before passing it to be fitted, which removes the extra dimension
"""
y_train = y_train.flatten()
mlr1.fit(X_train, y_train)

Epoch: 0, loss: 1662.0009063444106
Epoch: 10, loss: 1.040334944197476e+118
Epoch: 20, loss: 9.39238038629739e+232
Epoch: 30, loss: inf
Epoch: 40, loss: inf
Epoch: 50, loss: inf
Epoch: 60, loss: nan
Epoch: 70, loss: nan
Epoch: 80, loss: nan
Epoch: 90, loss: nan


/var/folders/z1/cd374qgx421_hlkd_bg_nxtr0000gn/T/ipykernel_73821/1406160549.py:10: RuntimeWarning:

overflow encountered in square

/var/folders/z1/cd374qgx421_hlkd_bg_nxtr0000gn/T/ipykernel_73821/1406160549.py:31: RuntimeWarning:

overflow encountered in dot

/Users/adrinorosario/Desktop/ml-kit/env/lib/python3.14/site-packages/numpy/_core/fromnumeric.py:83: RuntimeWarning:

overflow encountered in reduce

/var/folders/z1/cd374qgx421_hlkd_bg_nxtr0000gn/T/ipykernel_73821/1406160549.py:25: RuntimeWarning:

invalid value encountered in dot



In [37]:
y_train.shape, y_train.flatten().shape

((331, 1), (331,))

In [46]:
fig = make_subplots(rows=2, cols=3, 
                    subplot_titles=df_X.columns,
                    horizontal_spacing=0.1, vertical_spacing=0.15)

for i, col_name in enumerate(df_X.columns):
    row = (i // 3) + 1
    col = (i % 3) + 1

    # fig.add_trace(
    #     go.Scattergl(
    #         x=df_X[col_name],
    #         y =df_y['Y house price of unit area'],
    #         mode="markers",
    #         marker=dict(size=5, color='blue', opacity=0.5),
    #         name=col_name
    #     ),
    #     row=row, col=col
    # )

    fig.add_trace(
        go.Histogram(
            x=df_X[col_name],
            y=df_y['Y house price of unit area'],
            name=col_name,
            opacity=0.5
        ),
        row=row, col=col
    )

fig.update_layout(
    title_text="Features vs House Price of all the features in the dataset",
    height=800, width=1000,
    showlegend=False)
fig.show()

The histograms were plotted to understand the distribution of each feature in relation to the dependent variable. As of now, we can see that X3 displays a right-skewed distribution. We can calculate the skewness coefficient and if that $> 0$, then we can consider applying a log transform to it.

In [56]:
ols_mod1 = sm.OLS(df_X['X3 distance to the nearest MRT station'].to_numpy(), df_y['Y house price of unit area'].to_numpy())
res1 = ols_mod1.fit()
print(res1.summary())

                                 OLS Regression Results                                
Dep. Variable:                      y   R-squared (uncentered):                   0.195
Model:                            OLS   Adj. R-squared (uncentered):              0.193
Method:                 Least Squares   F-statistic:                              100.1
Date:                Fri, 13 Mar 2026   Prob (F-statistic):                    2.97e-21
Time:                        16:51:25   Log-Likelihood:                         -3612.8
No. Observations:                 414   AIC:                                      7228.
Df Residuals:                     413   BIC:                                      7232.
Df Model:                           1                                                  
Covariance Type:            nonrobust                                                  
                 coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------

In [60]:
px.histogram(np.log(df_X['X3 distance to the nearest MRT station'].to_numpy()))

In [63]:
log_transformed_X3 = np.log(df_X['X3 distance to the nearest MRT station'].to_numpy())

ols_mod2 = sm.OLS(log_transformed_X3, df_y['Y house price of unit area'].to_numpy())
res2 = ols_mod2.fit()
print(res2.summary())

                                 OLS Regression Results                                
Dep. Variable:                      y   R-squared (uncentered):                   0.783
Model:                            OLS   Adj. R-squared (uncentered):              0.782
Method:                 Least Squares   F-statistic:                              1489.
Date:                Fri, 13 Mar 2026   Prob (F-statistic):                   4.69e-139
Time:                        16:56:27   Log-Likelihood:                         -1045.5
No. Observations:                 414   AIC:                                      2093.
Df Residuals:                     413   BIC:                                      2097.
Df Model:                           1                                                  
Covariance Type:            nonrobust                                                  
                 coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------

In [69]:
df_X = df_X.with_columns(
    pl.Series("X3 distance to the nearest MRT station", log_transformed_X3)
)

In [71]:
px.histogram(df_X['X3 distance to the nearest MRT station'].to_numpy())